# WiFi CSI — Fine-Tuning on Captures Data
**Runtime**: Change to **T4 GPU** (Runtime > Change runtime type)

**Prerequisite**: Run `captures_cleaning.ipynb` first to create the cleaned parquet files.

This notebook fine-tunes:
1. **People Counting** — ResNet CNN (99.89%) on captures (0-3 people)
2. **Localization** — MultiScale + Transformer + SE-ResNet ensemble on captures (18 positions)

In [ ]:
!pip install -q pyarrow

import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time, gc, os, json, shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/fyp'
SAVE_DIR = '/content/drive/MyDrive/fyp'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy model files to local for fast loading
for f in ['best_resnet.pt', 'best_multiscale.pt', 'best_transformer.pt', 'best_se_resnet.pt']:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join('/content', f)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Copied {f} to local')
print('Setup done.')

---
## Part 1: People Counting Fine-Tuning

In [ ]:
# ============================================================
# Load people counting data (256 subcarriers, interpolated)
# ============================================================

pc_path = f'{DRIVE_DIR}/captures_people_counting.parquet'
amp_cols = [f'amp_{i}' for i in range(256)]
phase_cols = [f'phase_{i}' for i in range(256)]
table = pq.read_table(pc_path, columns=amp_cols + phase_cols + ['num_people'])

y_pc = table.column('num_people').to_numpy().astype(np.int64)
X_pc = np.empty((len(y_pc), 512), dtype=np.float32)
for i, col in enumerate(amp_cols + phase_cols):
    X_pc[:, i] = table.column(col).to_numpy().astype(np.float32)
del table; gc.collect()

print(f'People counting data: {X_pc.shape}')
print(f'Classes: {dict(zip(*np.unique(y_pc, return_counts=True)))}')

# Normalize using captures statistics
pc_amp_mean = X_pc[:, :256].mean(0)
pc_amp_std = X_pc[:, :256].std(0) + 1e-8
pc_phase_mean = X_pc[:, 256:].mean(0)
pc_phase_std = X_pc[:, 256:].std(0) + 1e-8
X_pc[:, :256] = (X_pc[:, :256] - pc_amp_mean) / pc_amp_std
X_pc[:, 256:] = (X_pc[:, 256:] - pc_phase_mean) / pc_phase_std

# Stratified split 70/15/15
idx = np.arange(len(y_pc))
idx_train, idx_temp = train_test_split(idx, test_size=0.3, random_state=42, stratify=y_pc)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42, stratify=y_pc[idx_temp])
del idx_temp

y_train = y_pc[idx_train]
class_counts = np.bincount(y_train, minlength=6)  # 6 classes (model has 6 outputs)
class_counts[class_counts == 0] = 1  # avoid div by zero for classes 4, 5
class_weights = len(y_train) / (6 * class_counts + 1)

print(f'\nTrain: {len(idx_train):,} | Val: {len(idx_val):,} | Test: {len(idx_test):,}')
print(f'Class weights: {np.round(class_weights, 2)}')

In [ ]:
# ============================================================
# Define ResNet CNN (EXACT copy from best_train.ipynb)
# ============================================================

class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(channels)
    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return torch.relu(x + residual)

class CSI_ResNet(nn.Module):
    def __init__(self, num_classes=6, dropout=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.layer1 = nn.Sequential(ResBlock(64), ResBlock(64), nn.MaxPool1d(2))
        self.expand1 = nn.Sequential(nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.layer2 = nn.Sequential(ResBlock(128), ResBlock(128), nn.MaxPool1d(2))
        self.expand2 = nn.Sequential(nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.layer3 = nn.Sequential(ResBlock(256), ResBlock(256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.expand1(self.layer1(x))
        x = self.expand2(self.layer2(x))
        x = self.pool(self.layer3(x))
        return self.classifier(x)

# Load pre-trained weights
pc_model = CSI_ResNet().to(device)
pc_model.load_state_dict(torch.load('/content/best_resnet.pt', weights_only=True, map_location=device))
print(f'Loaded pre-trained ResNet CNN ({sum(p.numel() for p in pc_model.parameters()):,} params)')

In [ ]:
# ============================================================
# Fine-tune people counting model
# Phase 1: lower LR, all layers trainable
# (early layers must adapt to interpolated 128->256 input)
# ============================================================

def make_pc_dataset(indices):
    d = X_pc[indices]
    x_2ch = np.stack([d[:, :256], d[:, 256:]], axis=1)  # (N, 2, 256)
    return TensorDataset(torch.tensor(x_2ch), torch.tensor(y_pc[indices]))

BS = 512
train_loader = DataLoader(make_pc_dataset(idx_train), batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(make_pc_dataset(idx_val), batch_size=BS*2, num_workers=2, pin_memory=True)
test_loader = DataLoader(make_pc_dataset(idx_test), batch_size=BS*2, num_workers=2, pin_memory=True)

# Weighted loss for class imbalance
weights_t = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_t)
optimizer = optim.AdamW(pc_model.parameters(), lr=5e-4, weight_decay=1e-3)
EPOCHS = 25
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler()

# Zero-shot baseline
pc_model.eval()
correct = 0
with torch.no_grad():
    for xb, yb in test_loader:
        correct += (pc_model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
print(f'Zero-shot accuracy (before fine-tuning): {correct/len(idx_test):.4f}')

# Training loop
best_val_acc = 0
hist = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    t0 = time.time()
    pc_model.train()
    tloss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            loss = criterion(pc_model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tloss += loss.item() * len(xb)
    tloss /= len(train_loader.dataset)
    scheduler.step()

    pc_model.eval()
    correct = 0
    with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
        for xb, yb in val_loader:
            correct += (pc_model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
    vacc = correct / len(idx_val)

    hist['train_loss'].append(tloss)
    hist['val_acc'].append(vacc)

    tag = ''
    if vacc > best_val_acc:
        best_val_acc = vacc
        torch.save(pc_model.state_dict(), f'{SAVE_DIR}/finetuned_resnet_people.pt')
        tag = ' *best*'

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | TL: {tloss:.4f} | VA: {vacc:.4f} | {time.time()-t0:.1f}s{tag}')

print(f'\nBest val accuracy: {best_val_acc:.4f}')

In [ ]:
# ============================================================
# Evaluate fine-tuned people counting model
# ============================================================

pc_model.load_state_dict(torch.load(f'{SAVE_DIR}/finetuned_resnet_people.pt', weights_only=True))
pc_model.eval()
preds, labels = [], []
with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
    for xb, yb in test_loader:
        preds.append(pc_model(xb.to(device)).argmax(1).cpu().numpy())
        labels.append(yb.numpy())
y_pred = np.concatenate(preds)
y_true = np.concatenate(labels)

acc = accuracy_score(y_true, y_pred)
print(f'=== Fine-tuned People Counting Test Accuracy: {acc:.4f} ===')
# Only show classes that exist in test set
present = sorted(set(y_true) | set(y_pred))
names = [f'{i} people' for i in present]
print(classification_report(y_true, y_pred, labels=present, target_names=names))

cm = confusion_matrix(y_true, y_pred, labels=present)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'{i}p' for i in present],
            yticklabels=[f'{i}p' for i in present])
plt.title(f'Fine-tuned People Counting (Acc: {acc:.4f})')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.show()

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist['train_loss']); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch'); ax1.grid(alpha=0.3)
ax2.plot(hist['val_acc'], color='green'); ax2.set_title('Val Accuracy'); ax2.set_xlabel('Epoch'); ax2.grid(alpha=0.3)
ax2.axhline(best_val_acc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_acc:.4f}')
ax2.legend()
plt.tight_layout(); plt.show()

# Save normalization stats for inference
np.savez(f'{SAVE_DIR}/finetuned_pc_norm_stats.npz',
         amp_mean=pc_amp_mean, amp_std=pc_amp_std,
         phase_mean=pc_phase_mean, phase_std=pc_phase_std)
print(f'Saved normalization stats to {SAVE_DIR}/finetuned_pc_norm_stats.npz')

---
## Part 2: Localization Fine-Tuning

In [ ]:
# ============================================================
# Load localization data (128 subcarriers, native)
# ============================================================

# Free people counting memory
del X_pc, y_pc, train_loader, val_loader, test_loader
gc.collect(); torch.cuda.empty_cache()

N_SUB = 128
loc_path = f'{DRIVE_DIR}/captures_localization.parquet'
amp_cols = [f'amp_{i}' for i in range(N_SUB)]
phase_cols = [f'phase_{i}' for i in range(N_SUB)]
table = pq.read_table(loc_path, columns=amp_cols + phase_cols + ['x', 'y', 'z', 'filename'])

X_loc = np.empty((len(table), N_SUB * 2), dtype=np.float32)
for i, col in enumerate(amp_cols + phase_cols):
    X_loc[:, i] = table.column(col).to_numpy().astype(np.float32)

y_x = table.column('x').to_numpy().astype(np.float32)
y_y = table.column('y').to_numpy().astype(np.float32)
y_z = table.column('z').to_numpy().astype(np.float32)
Y_loc = np.column_stack([y_x, y_y, y_z])
filenames = table.column('filename').to_pylist()
del table; gc.collect()

print(f'Localization data: {X_loc.shape}')
print(f'Target ranges: x=[{y_x.min():.1f}, {y_x.max():.1f}], y=[{y_y.min():.1f}, {y_y.max():.1f}], z=[{y_z.min():.1f}, {y_z.max():.1f}]')

# File-level train/val/test split to avoid data leakage
unique_files = sorted(set(filenames))
np.random.seed(42)
np.random.shuffle(unique_files)
n_files = len(unique_files)
n_train = int(0.7 * n_files)  # ~12-13 files
n_val = int(0.15 * n_files)   # ~2-3 files
train_files = set(unique_files[:n_train])
val_files = set(unique_files[n_train:n_train + n_val])
test_files = set(unique_files[n_train + n_val:])

file_arr = np.array(filenames)
idx_train = np.where(np.isin(file_arr, list(train_files)))[0]
idx_val = np.where(np.isin(file_arr, list(val_files)))[0]
idx_test = np.where(np.isin(file_arr, list(test_files)))[0]

print(f'\nFile-level split:')
print(f'  Train: {len(train_files)} files, {len(idx_train):,} rows')
print(f'  Val:   {len(val_files)} files, {len(idx_val):,} rows')
print(f'  Test:  {len(test_files)} files, {len(idx_test):,} rows')
print(f'  Train files: {sorted(train_files)}')
print(f'  Val files:   {sorted(val_files)}')
print(f'  Test files:  {sorted(test_files)}')

# Normalize features using training set stats
loc_X_mean = X_loc[idx_train].mean(0)
loc_X_std = X_loc[idx_train].std(0) + 1e-8
X_loc = (X_loc - loc_X_mean) / loc_X_std

# Normalize targets using training set stats
loc_Y_mean = Y_loc[idx_train].mean(0)
loc_Y_std = Y_loc[idx_train].std(0) + 1e-8
Y_loc_norm = (Y_loc - loc_Y_mean) / loc_Y_std

print(f'\nTarget stats: mean={loc_Y_mean}, std={loc_Y_std}')

In [ ]:
# ============================================================
# Define ALL localization model architectures
# (EXACT copies from localization_v2.ipynb)
# ============================================================

# --- Model 1: Multi-Scale CNN ---
class MultiScaleCNN(nn.Module):
    def __init__(self, n_sub=128, dropout=0.3):
        super().__init__()
        self.branch_fine = nn.Sequential(
            nn.Conv1d(2, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.branch_mid = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.branch_wide = nn.Sequential(
            nn.Conv1d(2, 64, 15, padding=7), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 15, padding=7), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.merge = nn.Sequential(
            nn.Conv1d(192, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 512, 3, padding=1), nn.BatchNorm1d(512), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(512, 512, 3, padding=1), nn.BatchNorm1d(512), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(), nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 3),
        )
    def forward(self, x):
        f = self.branch_fine(x)
        m = self.branch_mid(x)
        w = self.branch_wide(x)
        x = torch.cat([f, m, w], dim=1)
        x = self.merge(x)
        return self.regressor(x)

# --- Model 2: Transformer ---
class CSI_Transformer(nn.Module):
    def __init__(self, n_sub=128, d_model=64, nhead=4, n_layers=4, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(2, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, n_sub, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.pool = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 3),
        )
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, 128, 2)
        x = self.input_proj(x) + self.pos_embed
        x = self.encoder(x)
        x = x.permute(0, 2, 1)  # (B, d_model, 128)
        x = self.pool(x)
        return self.regressor(x)

# --- Model 3: Deep SE-ResNet ---
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid(),
        )
    def forward(self, x):
        w = self.fc(x).unsqueeze(-1)
        return x * w

class SEResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(channels)
        self.se = SEBlock(channels)
    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = self.se(x)
        return torch.relu(x + residual)

class DeepSEResNet(nn.Module):
    def __init__(self, n_sub=128, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.layer1 = nn.Sequential(SEResBlock(64), SEResBlock(64), SEResBlock(64), nn.MaxPool1d(2))
        self.expand1 = nn.Sequential(nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.layer2 = nn.Sequential(SEResBlock(128), SEResBlock(128), SEResBlock(128), nn.MaxPool1d(2))
        self.expand2 = nn.Sequential(nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.layer3 = nn.Sequential(SEResBlock(256), SEResBlock(256), SEResBlock(256), nn.MaxPool1d(2))
        self.expand3 = nn.Sequential(nn.Conv1d(256, 512, 1), nn.BatchNorm1d(512), nn.ReLU())
        self.layer4 = nn.Sequential(SEResBlock(512), SEResBlock(512))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.regressor = nn.Sequential(
            nn.Flatten(), nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 3),
        )
    def forward(self, x):
        x = self.stem(x)
        x = self.expand1(self.layer1(x))
        x = self.expand2(self.layer2(x))
        x = self.expand3(self.layer3(x))
        x = self.pool(self.layer4(x))
        return self.regressor(x)

print('All model architectures defined.')

In [ ]:
# ============================================================
# Fine-tune all 3 localization models
# ============================================================

# Data augmentation dataset
class CSILocDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y, augment=False):
        self.X = torch.tensor(X)
        self.Y = torch.tensor(Y)
        self.augment = augment
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        x = self.X[idx].view(2, N_SUB)  # (2, 128)
        y = self.Y[idx]
        if self.augment:
            x = x + torch.randn_like(x) * 0.05  # Gaussian noise
            mask = torch.rand(N_SUB) > 0.05      # 5% subcarrier masking
            x = x * mask.unsqueeze(0)
        return x, y

train_ds = CSILocDataset(X_loc[idx_train], Y_loc_norm[idx_train], augment=True)
val_ds = CSILocDataset(X_loc[idx_val], Y_loc_norm[idx_val], augment=False)
test_ds = CSILocDataset(X_loc[idx_test], Y_loc_norm[idx_test], augment=False)

BS = 512
loc_train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
loc_val_loader = DataLoader(val_ds, batch_size=BS*2, num_workers=2, pin_memory=True)
loc_test_loader = DataLoader(test_ds, batch_size=BS*2, num_workers=2, pin_memory=True)


def finetune_loc_model(model, name, pt_file, epochs=30, lr=5e-4):
    """Fine-tune a single localization model."""
    # Load pre-trained weights
    model.load_state_dict(torch.load(f'/content/{pt_file}', weights_only=True, map_location=device))
    model.to(device)
    print(f'\n{"="*50}')
    print(f'Fine-tuning {name} ({sum(p.numel() for p in model.parameters()):,} params)')
    print(f'{"="*50}')

    criterion = nn.HuberLoss(delta=1.0)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler()

    best_val_mae = float('inf')
    save_path = f'{SAVE_DIR}/finetuned_{name}.pt'

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        tloss = 0
        for xb, yb in loc_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda'):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            tloss += loss.item() * len(xb)
        tloss /= len(loc_train_loader.dataset)
        scheduler.step()

        model.eval()
        all_p, all_l = [], []
        with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
            for xb, yb in loc_val_loader:
                out = model(xb.to(device)).cpu().numpy()
                all_p.append(out * loc_Y_std + loc_Y_mean)
                all_l.append(yb.numpy() * loc_Y_std + loc_Y_mean)
        vp, vl = np.concatenate(all_p), np.concatenate(all_l)
        vmae_3d = np.sqrt(((vp - vl)**2).sum(axis=1)).mean()

        tag = ''
        if vmae_3d < best_val_mae:
            best_val_mae = vmae_3d
            torch.save(model.state_dict(), save_path)
            tag = ' *best*'

        if (epoch + 1) % 5 == 0 or tag:
            print(f'  Ep {epoch+1:2d}/{epochs} | TL: {tloss:.4f} | 3D MAE: {vmae_3d:.3f}m | {time.time()-t0:.1f}s{tag}')

    print(f'  Best val 3D MAE: {best_val_mae:.4f}m')
    # Reload best
    model.load_state_dict(torch.load(save_path, weights_only=True))
    return model, best_val_mae


# Fine-tune all three
model1 = MultiScaleCNN(n_sub=N_SUB)
model1, mae1 = finetune_loc_model(model1, 'multiscale_loc', 'best_multiscale.pt', epochs=30)

model2 = CSI_Transformer(n_sub=N_SUB)
model2, mae2 = finetune_loc_model(model2, 'transformer_loc', 'best_transformer.pt', epochs=30, lr=3e-4)

model3 = DeepSEResNet(n_sub=N_SUB)
model3, mae3 = finetune_loc_model(model3, 'se_resnet_loc', 'best_se_resnet.pt', epochs=30)

In [ ]:
# ============================================================
# Evaluate localization + ensemble
# ============================================================

def eval_loc_model(model, name):
    model.eval()
    preds, labels = [], []
    with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
        for xb, yb in loc_test_loader:
            out = model(xb.to(device)).cpu().numpy()
            preds.append(out * loc_Y_std + loc_Y_mean)
            labels.append(yb.numpy() * loc_Y_std + loc_Y_mean)
    p, l = np.concatenate(preds), np.concatenate(labels)
    metrics = {}
    for i, coord in enumerate(['x', 'y', 'z']):
        metrics[coord] = {
            'mae': float(mean_absolute_error(l[:, i], p[:, i])),
            'rmse': float(np.sqrt(mean_squared_error(l[:, i], p[:, i]))),
            'r2': float(r2_score(l[:, i], p[:, i])),
        }
    dist = np.sqrt(((p - l)**2).sum(axis=1))
    metrics['3d_dist'] = {
        'mae': float(dist.mean()),
        'median': float(np.median(dist)),
        'p90': float(np.percentile(dist, 90)),
    }
    print(f'\n=== {name} ===')
    for c in ['x', 'y', 'z']:
        print(f'  {c}: MAE={metrics[c]["mae"]:.4f}m  R2={metrics[c]["r2"]:.4f}')
    print(f'  3D: mean={metrics["3d_dist"]["mae"]:.4f}m  median={metrics["3d_dist"]["median"]:.4f}m  p90={metrics["3d_dist"]["p90"]:.4f}m')
    return p, l, metrics

preds1, labels1, m1 = eval_loc_model(model1, 'Multi-Scale CNN')
preds2, labels2, m2 = eval_loc_model(model2, 'Transformer')
preds3, labels3, m3 = eval_loc_model(model3, 'Deep SE-ResNet')

# Ensemble: inverse-MAE weighted average
w1 = 1.0 / m1['3d_dist']['mae']
w2 = 1.0 / m2['3d_dist']['mae']
w3 = 1.0 / m3['3d_dist']['mae']
wt = w1 + w2 + w3
ens_preds = (w1 * preds1 + w2 * preds2 + w3 * preds3) / wt
Y_test = labels1

ens_dist = np.sqrt(((ens_preds - Y_test)**2).sum(axis=1))
print(f'\n=== Ensemble ===')
print(f'  Weights: MultiScale={w1/wt:.3f}, Transformer={w2/wt:.3f}, SE-ResNet={w3/wt:.3f}')
for i, coord in enumerate(['x', 'y', 'z']):
    mae_c = float(mean_absolute_error(Y_test[:, i], ens_preds[:, i]))
    print(f'  {coord}: MAE={mae_c:.4f}m')
print(f'  3D: mean={ens_dist.mean():.4f}m  median={np.median(ens_dist):.4f}m  p90={np.percentile(ens_dist, 90):.4f}m')

# Comparison bar chart
all_results = {
    'Multi-Scale CNN': m1['3d_dist']['mae'],
    'Transformer': m2['3d_dist']['mae'],
    'Deep SE-ResNet': m3['3d_dist']['mae'],
    'Ensemble': float(ens_dist.mean()),
}
plt.figure(figsize=(8, 4))
plt.barh(list(all_results.keys()), list(all_results.values()), color=['steelblue', 'steelblue', 'steelblue', 'green'])
plt.xlabel('3D Distance Error (m)')
plt.title('Fine-tuned Localization Model Comparison')
for i, (name, v) in enumerate(all_results.items()):
    plt.text(v + 0.02, i, f'{v:.3f}m', va='center')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Save everything
# ============================================================

# Save localization normalization stats + ensemble weights
np.savez(f'{SAVE_DIR}/finetuned_loc_norm_stats.npz',
         X_mean=loc_X_mean, X_std=loc_X_std,
         Y_mean=loc_Y_mean, Y_std=loc_Y_std)

# Save all results
results = {
    'people_counting': {
        'model': 'ResNet CNN (fine-tuned)',
        'test_accuracy': float(acc),
        'best_val_accuracy': float(best_val_acc),
        'saved_as': 'finetuned_resnet_people.pt',
        'norm_stats': 'finetuned_pc_norm_stats.npz',
        'input_subcarriers': 256,
        'note': 'interpolated from 128 ESP32 subcarriers',
    },
    'localization': {
        'models': {
            'Multi-Scale CNN': {'3d_mae': m1['3d_dist']['mae'], 'saved_as': 'finetuned_multiscale_loc.pt'},
            'Transformer': {'3d_mae': m2['3d_dist']['mae'], 'saved_as': 'finetuned_transformer_loc.pt'},
            'Deep SE-ResNet': {'3d_mae': m3['3d_dist']['mae'], 'saved_as': 'finetuned_se_resnet_loc.pt'},
            'Ensemble': {'3d_mae': float(ens_dist.mean())},
        },
        'ensemble_weights': {'multiscale': w1/wt, 'transformer': w2/wt, 'se_resnet': w3/wt},
        'norm_stats': 'finetuned_loc_norm_stats.npz',
        'input_subcarriers': 128,
        'best_model': min(all_results, key=all_results.get),
    },
}
with open(f'{SAVE_DIR}/finetune_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)

print(f'\n{"="*60}')
print(f'ALL SAVED to {SAVE_DIR}/')
print(f'{"="*60}')
for fname in sorted(os.listdir(SAVE_DIR)):
    if 'finetune' in fname:
        size = os.path.getsize(os.path.join(SAVE_DIR, fname))
        unit = 'MB' if size > 1e6 else 'KB'
        val = size/1e6 if size > 1e6 else size/1e3
        print(f'  {fname} ({val:.1f} {unit})')

print(f'\n=== SUMMARY ===')
print(json.dumps(results, indent=2, default=float))